# Проект спринта 19. Модуль расчёта батч-признаков для платформы интернет-торговли.
## 1. Установка и импорт библиотек

In [72]:
%pip install -q dotenv
%pip install -q -U numpy==1.26.4 scipy numba matplotlib
%pip install -q pandas==2.2.3
%pip install -q -U sqlalchemy
%pip install -q -U psycopg2-binary

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/766

In [83]:
# Системные библиотеки
import os
from dotenv import load_dotenv
import math
import time

# Библиотека для доступа к СУБД
from sqlalchemy import create_engine, text

# Библиотеки для вычислений и работы с датасетами
import pandas as pd
import numpy as np

# Для форматирования в HTML
from IPython.display import display, HTML, Markdown

### Определение констант

In [ ]:
RUN_DATE = "2024-06-01"

# Интервал дат не будет включать дату запуска DAG, поэтому вычитаем 1 день
run_date = pd.to_datetime(RUN_DATE)

## 2. Загрузка параметров и соединение с БД

In [75]:
load_dotenv(".env")

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    os.getenv("DB_USER"),
    os.getenv("DB_PASSWORD"),
    os.getenv("DB_HOST"),
    os.getenv("DB_PORT"),
    os.getenv("DB_NAME")
)

# Создание пула соединений
engine = create_engine(connection_string) 

## 3. Загрузка необработанных данных

In [ ]:
def show_dataset_info(df, title):
    display(HTML(f"<h2>{title}</h2>"))
    df.info()
    display(df.head())
    display(HTML("<h3>Даты с</h3>"))
    display(pd.DataFrame({"Начальная дата": df.select_dtypes(include='datetime64').min(),
             "Конечная дата": df.select_dtypes(include='datetime64').max()}))

    display(HTML(f"<h3>Количество полных дубликатов: {df.duplicated().sum()}</h3>"))

def load_data():
    with engine.connect() as connection:
        df_events = pd.read_sql_table('events', con=connection)
        df_sessions = pd.read_sql_table('sessions', con=connection)
        df_orders = pd.read_sql_table('orders', con=connection)

        return df_events, df_sessions, df_orders
        

def load_data_by_run_date(run_date: pd.Timestamp):
    evt_query = text("""
        SELECT customer_id, events.*
        FROM 
            public.events
            INNER JOIN public.sessions USING (session_id)
        WHERE DATE_TRUNC('day', timestamp) BETWEEN :run_date - 30 AND :run_date
    """)
    session_query = text("""
        SELECT *
        FROM public.sessions
        WHERE DATE_TRUNC('day', start_time) BETWEEN :run_date - 30 AND :run_date
    """)
    orders_query = text("""
        SELECT *
        FROM public.orders
        WHERE DATE_TRUNC('day', order_time) BETWEEN :run_date - 30 AND :run_date
    """)

    with engine.connect() as connection:
        df_events = pd.read_sql(evt_query, con=connection, params={"run_date": run_date.date()})
        df_sessions = pd.read_sql(session_query, con=connection, params={"run_date": run_date.date()})
        df_orders = pd.read_sql(orders_query, con=connection, params={"run_date": run_date.date()})

        
        return {
            "events_30": df_events,
            "sessions_30": df_sessions,
            "orders_30": df_orders,
            "events_7": df_events[df_events['timestamp'] >= run_date - pd.Timedelta(days=7)],
            "sessions_7": df_sessions[df_sessions['start_time'] >= run_date - pd.Timedelta(days=7)],
            "orders_7": df_orders[df_orders['order_time'] >= run_date - pd.Timedelta(days=7)]
        }   

load_data()
data = load_data_by_run_date(run_date)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239089 entries, 0 to 239088
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   event_id      239089 non-null  int64         
 1   session_id    239089 non-null  int64         
 2   timestamp     239089 non-null  datetime64[ns]
 3   event_type    239089 non-null  object        
 4   product_id    214208 non-null  float64       
 5   qty           44908 non-null   float64       
 6   cart_size     14229 non-null   float64       
 7   payment       10652 non-null   object        
 8   discount_pct  10652 non-null   float64       
 9   amount_usd    10652 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(2), object(2)
memory usage: 18.2+ MB


,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,11,2,2025-01-31 21:48:42,page_view,929.0,NaN,NaN,None,NaN,NaN
1,12,2,2025-01-31 21:57:42,page_view,226.0,NaN,NaN,None,NaN,NaN
2,13,2,2025-01-31 21:59:07,add_to_cart,226.0,1.0,NaN,None,NaN,NaN
3,14,2,2025-01-31 22:01:42,page_view,864.0,NaN,NaN,None,NaN,NaN
4,15,2,2025-01-31 22:25:42,page_view,1151.0,NaN,NaN,None,NaN,NaN


,Начальная дата,Конечная дата
timestamp,2024-01-01 01:02:40,2025-11-01 01:50:04


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37699 entries, 0 to 37698
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   session_id   37699 non-null  int64         
 1   customer_id  37699 non-null  int64         
 2   start_time   37699 non-null  datetime64[ns]
 3   device       37699 non-null  object        
 4   source       37699 non-null  object        
 5   country      37699 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 1.7+ MB


,session_id,customer_id,start_time,device,source,country
0,2,13917,2025-01-31 21:29:42,desktop,organic,PL
1,3,1022,2024-02-19 00:52:50,tablet,organic,FR
2,4,2882,2024-08-04 19:54:31,mobile,direct,GB
3,9,6476,2024-02-22 16:57:44,mobile,direct,PL
4,10,6145,2024-12-04 18:53:13,mobile,organic,US


,Начальная дата,Конечная дата
start_time,2024-01-01 00:57:40,2025-10-31 23:34:11


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10652 entries, 0 to 10651
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        10652 non-null  int64         
 1   customer_id     10652 non-null  int64         
 2   order_time      10652 non-null  datetime64[ns]
 3   payment_method  10652 non-null  object        
 4   discount_pct    10652 non-null  float64       
 5   subtotal_usd    10652 non-null  float64       
 6   total_usd       10652 non-null  float64       
 7   country         10652 non-null  object        
 8   device          10652 non-null  object        
 9   source          10652 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 832.3+ KB


,order_id,customer_id,order_time,payment_method,discount_pct,subtotal_usd,total_usd,country,device,source
0,1,13917,2025-01-31 23:07:42,card,20.0,107.15,85.72,PL,desktop,organic
1,2,1022,2024-02-19 01:17:50,card,0.0,116.17,116.17,FR,tablet,organic
2,3,6145,2024-12-04 20:24:13,card,0.0,137.35,137.35,US,mobile,organic
3,4,3152,2024-07-17 08:50:47,card,15.0,32.18,27.35,BR,mobile,email
4,11,14044,2025-03-12 02:48:39,card,15.0,55.77,47.40,US,mobile,direct


,Начальная дата,Конечная дата
order_time,2024-01-01 07:08:06,2025-10-31 22:59:41


- Записи в таблицах присутствуют для дат 01.01.2024 по 31.10.2025 включительно.
- Даты для формирования 30-дневных срезов - с 31.01.2024.
- Целевая переменная может быть сформирована для данных до 24.10.2025 включительно.
- Пропуски имеются только в таблице `events`, для нас важны только записи с непустым `product_id`.

## 4. Создание батч-признаков

In [ ]:
# Предобработка загруженных данных
def preprocess(df_events, df_sessions, df_orders):
    # Удаляем строки без product_id
    df_events = df_events[df_events['product_id'].notna()]

    return df_events, df_sessions, df_orders

# Получение 7- и 30-дневных срезов
def get_slices(run_date, df_events, df_sessions, df_orders):
    df_dict = {}

    date_from_7 = run_date - pd.Timedelta(days=7)
    date_from_30 = run_date - pd.Timedelta(days=30)

    if date_from_30 < df_events['timestamp'].min():
        raise ValueError("Slice for this run_date is not available")

    df_dict['events_7'] = (df_events[(df_events['timestamp'] >= date_from_7) &
                                     (df_events['timestamp'] < run_date)]
                           .merge(df_sessions[['session_id', 'customer_id']], on='session_id'))
    df_dict['events_30'] = (df_events[(df_events['timestamp'] >= date_from_30) &
                                      (df_events['timestamp'] < run_date)]
                           .merge(df_sessions[['session_id', 'customer_id']], on='session_id'))
    
    df_dict['sessions_7'] = df_sessions[(df_sessions['start_time'] >= date_from_7) &
                                        (df_sessions['start_time'] < run_date)]
    df_dict['sessions_30'] = df_sessions[(df_sessions['start_time'] >= date_from_30) &
                                         (df_sessions['start_time'] < run_date)]
    df_dict['orders_7'] = df_orders[(df_orders['order_time'] >= date_from_7) &
                                    (df_orders['order_time'] < run_date)]
    df_dict['orders_30'] = df_orders[(df_orders['order_time'] >= date_from_30) &
                                     (df_orders['order_time'] < run_date)]

    return df_dict

# Создание батч-признаков
def create_batch_features(df_dict):
    df = {}

    df['page_views_7_count'] = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'page_view']
                                .groupby('customer_id')
                                .size()
                                .rename('page_views_7_count'))

    df['page_views_30_count'] = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'page_view']
                                .groupby('customer_id')
                                .size()
                                .rename('page_views_30_count'))

    df['add_to_cart_7_count'] = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'add_to_cart']
                                .groupby('customer_id')
                                .size()
                                .rename('add_to_cart_7_count'))
    
    df['add_to_cart_30_count'] = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'add_to_cart']
                                .groupby('customer_id')
                                .size()
                                .rename('add_to_cart_30_count'))

    df_purchase_count_7 = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'purchase']
                                .groupby('customer_id')
                                .size()
                                .rename('purchase_count_7'))
    
    df_purchase_count_30 = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'purchase']
                                .groupby('customer_id')
                                .size()
                                .rename('purchase_count_30'))
    
    df['cart_conv_7'] = (df['add_to_cart_7_count'] / df['page_views_7_count']).rename('cart_conv_7')
    df['cart_conv_30'] = (df['add_to_cart_30_count'] / df['page_views_30_count']).rename('cart_conv_30')
    df['purchase_conv_7'] = (df_purchase_count_7 / df['add_to_cart_7_count']).rename('purchase_conv_7')
    df['purchase_conv_30'] = (df_purchase_count_30 / df['add_to_cart_30_count']).rename('purchase_conv_30')
    df['unique_products_7'] = df_dict['events_7'].groupby('customer_id')['product_id'].nunique().rename('unique_products_7')
    df['unique_products_30'] = df_dict['events_30'].groupby('customer_id')['product_id'].nunique().rename('unique_products_30')

    df_session_durations = (df_dict['sessions_30']
                            .merge(df_dict['events_30']
                                      .groupby('session_id')['timestamp']
                                      .max()
                                      .rename('last_event_time'), 
                                    on='session_id', how='left'))
    df_session_durations['session_duration'] = (df_session_durations['last_event_time'] - 
                                                df_session_durations['start_time']).dt.total_seconds()                        
    
    df['mean_session_duration_30'] = (df_session_durations
                                        .groupby('customer_id')['session_duration']
                                        .mean()
                                        .rename('mean_session_duration_30'))
    df['sessions_count_7'] = df_dict['sessions_7'].groupby('customer_id').size().rename('sessions_count_7')
    df['sessions_count_30'] = df_dict['sessions_30'].groupby('customer_id').size().rename('sessions_count_30')

    df_last_purchase = (
        df_dict['events_7']
        .loc[df_dict['events_7']['event_type'] == 'purchase']
        .groupby('customer_id')['timestamp']
        .max()
    )

    df['days_from_last_purchase'] = (
        (run_date.normalize() - df_last_purchase.dt.normalize()).dt.days + 1
    ).rename('days_from_last_purchase')

    df['orders_count_30'] = df_dict['orders_30'].groupby('customer_id').size().rename('orders_count_30')
    df['total_usd_sum_30'] = df_dict['orders_30'].groupby('customer_id')['total_usd'].sum().rename('total_usd_sum_30')
    df['total_usd_mean_30'] = df_dict['orders_30'].groupby('customer_id')['total_usd'].mean().rename('total_usd_mean_30')

    # Объединение всех признаков в один DataFrame
    batch_features = pd.concat(df.values(), axis=1)

    batch_features.fillna({'days_from_last_purchase': -1}, inplace=True)
    batch_features.fillna(0, inplace=True)

    batch_features.info()
    return batch_features.reset_index()

create_batch_features(get_slices(run_date, *preprocess(*load_data())))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239089 entries, 0 to 239088
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   event_id      239089 non-null  int64         
 1   session_id    239089 non-null  int64         
 2   timestamp     239089 non-null  datetime64[ns]
 3   event_type    239089 non-null  object        
 4   product_id    214208 non-null  float64       
 5   qty           44908 non-null   float64       
 6   cart_size     14229 non-null   float64       
 7   payment       10652 non-null   object        
 8   discount_pct  10652 non-null   float64       
 9   amount_usd    10652 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(2), object(2)
memory usage: 18.2+ MB


,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,11,2,2025-01-31 21:48:42,page_view,929.0,NaN,NaN,None,NaN,NaN
1,12,2,2025-01-31 21:57:42,page_view,226.0,NaN,NaN,None,NaN,NaN
2,13,2,2025-01-31 21:59:07,add_to_cart,226.0,1.0,NaN,None,NaN,NaN
3,14,2,2025-01-31 22:01:42,page_view,864.0,NaN,NaN,None,NaN,NaN
4,15,2,2025-01-31 22:25:42,page_view,1151.0,NaN,NaN,None,NaN,NaN


,Начальная дата,Конечная дата
timestamp,2024-01-01 01:02:40,2025-11-01 01:50:04


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37699 entries, 0 to 37698
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   session_id   37699 non-null  int64         
 1   customer_id  37699 non-null  int64         
 2   start_time   37699 non-null  datetime64[ns]
 3   device       37699 non-null  object        
 4   source       37699 non-null  object        
 5   country      37699 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 1.7+ MB


,session_id,customer_id,start_time,device,source,country
0,2,13917,2025-01-31 21:29:42,desktop,organic,PL
1,3,1022,2024-02-19 00:52:50,tablet,organic,FR
2,4,2882,2024-08-04 19:54:31,mobile,direct,GB
3,9,6476,2024-02-22 16:57:44,mobile,direct,PL
4,10,6145,2024-12-04 18:53:13,mobile,organic,US


...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10652 entries, 0 to 10651
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        10652 non-null  int64         
 1   customer_id     10652 non-null  int64         
 2   order_time      10652 non-null  datetime64[ns]
 3   payment_method  10652 non-null  object        
 4   discount_pct    10652 non-null  float64       
 5   subtotal_usd    10652 non-null  float64       
 6   total_usd       10652 non-null  float64       
 7   country         10652 non-null  object        
 8   device          10652 non-null  object        
 9   source          10652 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 832.3+ KB


,order_id,customer_id,order_time,payment_method,discount_pct,subtotal_usd,total_usd,country,device,source
0,1,13917,2025-01-31 23:07:42,card,20.0,107.15,85.72,PL,desktop,organic
1,2,1022,2024-02-19 01:17:50,card,0.0,116.17,116.17,FR,tablet,organic
2,3,6145,2024-12-04 20:24:13,card,0.0,137.35,137.35,US,mobile,organic
3,4,3152,2024-07-17 08:50:47,card,15.0,32.18,27.35,BR,mobile,email
4,11,14044,2025-03-12 02:48:39,card,15.0,55.77,47.40,US,mobile,direct


,Начальная дата,Конечная дата
order_time,2024-01-01 07:08:06,2025-10-31 22:59:41


<class 'pandas.core.frame.DataFrame'>
Index: 1591 entries, 47 to 8949
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   page_views_7_count        1591 non-null   float64
 1   page_views_30_count       1591 non-null   float64
 2   add_to_cart_7_count       1591 non-null   float64
 3   add_to_cart_30_count      1591 non-null   float64
 4   cart_conv_7               1591 non-null   float64
 5   cart_conv_30              1591 non-null   float64
 6   purchase_conv_7           1591 non-null   float64
 7   purchase_conv_30          1591 non-null   float64
 8   unique_products_7         1591 non-null   float64
 9   unique_products_30        1591 non-null   float64
 10  mean_session_duration_30  1591 non-null   float64
 11  sessions_count_7          1591 non-null   float64
 12  sessions_count_30         1591 non-null   float64
 13  days_from_last_purchase   1591 non-null   float64
 14  orders_count

,customer_id,page_views_7_count,page_views_30_count,add_to_cart_7_count,add_to_cart_30_count,cart_conv_7,cart_conv_30,purchase_conv_7,purchase_conv_30,unique_products_7,unique_products_30,mean_session_duration_30,sessions_count_7,sessions_count_30,days_from_last_purchase,orders_count_30,total_usd_sum_30,total_usd_mean_30
0,47,7.0,7.0,3.0,3.0,0.428571,0.428571,0.0,0.0,7.0,7.0,4987.0,1.0,1.0,-1.0,0.0,0.00,0.00
1,220,1.0,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0,1.0,1800.0,1.0,1.0,-1.0,0.0,0.00,0.00
2,246,1.0,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0,1.0,1740.0,1.0,1.0,-1.0,0.0,0.00,0.00
3,265,8.0,8.0,2.0,2.0,0.250000,0.250000,0.0,0.0,8.0,8.0,7140.0,1.0,1.0,-1.0,1.0,792.88,792.88
4,274,8.0,8.0,2.0,2.0,0.250000,0.250000,0.0,0.0,8.0,8.0,5327.0,1.0,1.0,-1.0,0.0,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586,19918,0.0,6.0,0.0,2.0,0.000000,0.333333,0.0,0.0,0.0,6.0,5340.0,0.0,1.0,-1.0,0.0,0.00,0.00
1587,19932,0.0,2.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,2.0,2580.0,0.0,1.0,-1.0,0.0,0.00,0.00
1588,19938,0.0,4.0,0.0,1.0,0.000000,0.250000,0.0,0.0,0.0,4.0,4287.0,0.0,1.0,-1.0,0.0,0.00,0.00
1589,19990,0.0,5.0,0.0,1.0,0.000000,0.200000,0.0,0.0,0.0,5.0,4020.0,0.0,1.0,-1.0,1.0,8.28,8.28
